# DO NOT USE PRERAINED MODELS

# NEOAI 2026: Creatures of the Static

---

# Overview

A catalog of **50 reference images** has reached us — exactly **10 per species**, 5 species in total. Then the signal arrived from elsewhere: **5000 snapshots from another dimension**, capturing the same creatures, but through a medium that distorts the image in unfamiliar ways. Your task: train on the clean catalog and correctly label every snapshot from the other side.

---

# Description

There are exactly **5 species** in the catalog. The training images came from our own world. The test snapshots, however, come from a different dimension. They carry the same creatures, but the medium they passed through has shifted them in ways the training. Knowing how exactly it differs is part of the puzzle — exploring the test set should be the first step.

---

# Data

All images are RGB, 32×32 pixels, PNG.

---

## train/ + train.csv

Clean reference images from our world, **50 total** — 10 per class.

`train.csv` columns:

- `id` — image filename inside `train/`
- `label` — integer class id `∈ {0, 1, 2, 3, 4}`

---

## test/ + test.csv

Snapshots from the other dimension, **5000 total** — 1000 per class.

`test.csv` columns:

- `id` — image filename inside `test/`

For every row, you must predict the class label.

The `test/` filenames are sequential (`00000.png … 04999.png`)

---

## sample_submission.csv

Template submission (all labels set to `0`).

Columns:

- `id` — must match `test.csv`
- `label` — predicted class id `∈ {0, 1, 2, 3, 4}`

---

# Task

For every row in `test.csv`, predict a single integer class label `∈ {0, 1, 2, 3, 4}`.

This is a **multi-class classification** problem with 5 balanced classes (1000 test samples per class).

---

# Evaluation

Submissions are evaluated using **Micro-F1 score**:

```text
F1_micro = TP_total / (TP_total + 0.5 * (FP_total + FN_total))
```

For a balanced 5-class task this is equivalent to plain accuracy. The metric is computed in two modes:

- **Public** — 50% of the test set (2500 samples), shown live during the competition,
- **Private** — the remaining 50% (2500 samples), revealed only after the deadline.

Both splits are stratified across all 5 classes.

---

# Submission File

The submission file must contain a header and follow this format:

```csv
id,label
00000.png,2
00001.png,4
00002.png,0
...
```

Exactly 5000 rows of predictions plus the header. The `id` column must match `test.csv` exactly (same filenames, same order is **not** required — the grader merges by `id`).



# НЕ ИСПЛЬЗУЙТЕ ПРЕТРЕЙН

# NEOAI 2026: Существа Статики

---

# Обзор

До нас дошёл каталог из **50 эталонных изображений** — ровно **по 10 на каждый вид**, всего 5 видов. Затем сигнал пришёл с другой стороны: **5000 снимков из другого измерения**, на которых запечатлены те же существа, но прошедшие через среду, искажающую изображения непривычным образом. Ваша задача — обучиться на чистом каталоге и правильно определить классы всех снимков с другой стороны.

---

# Описание

В каталоге присутствует ровно **5 видов существ**. Обучающие изображения были получены в нашем мире. Тестовые снимки, однако, пришли из другого измерения. На них изображены те же существа, но среда, через которую прошёл сигнал, изменила изображения неизвестным образом. Понять, чем именно отличается тестовый набор, — часть задачи, поэтому изучение тестовых данных должно стать вашим первым шагом.

---

# Данные

Все изображения имеют формат RGB, размер 32×32 пикселя, PNG.

---

## train/ + train.csv

Чистые эталонные изображения из нашего мира, **всего 50** — по 10 на каждый класс.

Колонки `train.csv`:

- `id` — имя файла изображения внутри `train/`
- `label` — целочисленный идентификатор класса `∈ {0, 1, 2, 3, 4}`

---

## test/ + test.csv

Снимки из другого измерения, **всего 5000** — по 1000 на каждый класс.

Колонки `test.csv`:

- `id` — имя файла изображения внутри `test/`

Для каждой строки необходимо предсказать метку класса.

Файлы в `test/` имеют последовательные имена (`00000.png … 04999.png`).

---

## sample_submission.csv

Шаблон файла отправки (все метки установлены в `0`).

Колонки:

- `id` — должен совпадать с `test.csv`
- `label` — предсказанный идентификатор класса `∈ {0, 1, 2, 3, 4}`

---

# Задача

Для каждой строки в `test.csv` предскажите одно целочисленное значение класса `∈ {0, 1, 2, 3, 4}`.

Это задача **многоклассовой классификации** с 5 сбалансированными классами (по 1000 тестовых объектов на каждый класс).

---

# Оценивание

Результаты оцениваются с помощью метрики **Micro-F1 score**:

```text
F1_micro = TP_total / (TP_total + 0.5 * (FP_total + FN_total))
```

Для сбалансированной задачи на 5 классов эта метрика эквивалентна обычной accuracy. Метрика вычисляется в двух режимах:

- **Public** — 50% тестового набора (2500 объектов), отображаются на лидерборде во время соревнования,
- **Private** — оставшиеся 50% тестового набора (2500 объектов), раскрываются только после завершения соревнования.

Обе части стратифицированы по всем 5 классам.

---

# Формат отправки

Файл отправки должен содержать заголовок и соответствовать следующему формату:

```csv
id,label
00000.png,2
00001.png,4
00002.png,0
...
```

Файл должен содержать ровно 5000 строк предсказаний плюс заголовок. Колонка `id` должна полностью совпадать с `test.csv` (те же имена файлов, порядок строк **необязателен** — система сопоставляет строки по `id`).

In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

np.random.seed(42)

In [2]:
def load_images_to_vectors(csv_file, images_folder):
    df = pd.read_csv(csv_file)
    features, labels, filenames = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img = Image.open(os.path.join(images_folder, row['id']))
        features.append(np.array(img).reshape(-1))
        filenames.append(row['id'])
        if 'label' in row:
            labels.append(row['label'])
    features = np.array(features)
    if labels:
        return features, np.array(labels), filenames
    return features, filenames    

In [3]:
X_train, y_train, _ = load_images_to_vectors('/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/train.csv', '/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/train')
X_test, _ = load_images_to_vectors('/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/test.csv', '/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/test')
print(f'train: {X_train.shape}  test: {X_test.shape}  classes: {len(np.unique(y_train))}')

100%|██████████| 5000/5000 [00:22<00:00, 219.41it/s]

train: (50, 3072)  test: (5000, 3072)  classes: 5


In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

X_val = X_tr[:10]
y_val = y_val[:10]

print(f'train: {X_tr.shape[0]}  val: {X_val.shape[0]}')

train: 40  val: 10


In [5]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print(f'val Micro-F1: {f1_score(y_val, val_pred, average="micro"):.4f}')

val Micro-F1: 0.2000


In [6]:
model.fit(X_train_s, y_train)
test_predictions = model.predict(X_test_s)

test_df = pd.read_csv('/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/test.csv')
test_df['label'] = test_predictions
test_df.to_csv('baseline_submission.csv', index=False)

In [23]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device="cuda" if torch.cuda.is_available() else "cpu"

TRAIN_CSV="/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/train.csv"
TEST_CSV="/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/test.csv"

TRAIN_DIR="/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/train"
TEST_DIR="/kaggle/input/competitions/neoai-2026-day-2-cv/compet_data/test"


def corrupt(img):

    img=img.copy()

    if random.random()<0.8:
        img=np.rot90(img, random.choice([0,1,2,3]))

    for _ in range(random.randint(1,3)):
        x=random.randint(0,31)
        img[:,x:x+1]=255

    if random.random()<0.5:
        x=random.randint(0,25)
        y=random.randint(0,25)
        s=random.randint(3,8)
        img[y:y+s,x:x+s]=255

    return img


def clean(img):

    gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)

    gray=cv2.medianBlur(gray,3)

    return gray/255.


X=[]
y=[]

train_df=pd.read_csv(TRAIN_CSV)

for _,row in train_df.iterrows():

    img=np.array(
        Image.open(
            os.path.join(TRAIN_DIR,row.id)
        ).convert("RGB")
    )

    for i in range(3000):

        aug=corrupt(img)

        X.append(clean(aug))
        y.append(row.label)


X=np.array(X,dtype=np.float32)
y=np.array(y)

print(X.shape)


X_train,X_val,y_train,y_val=train_test_split(
    X,y,
    test_size=0.1,
    stratify=y,
    random_state=42
)


class DS(Dataset):

    def __init__(self,X,y):
        self.X=torch.tensor(X).unsqueeze(1)
        self.y=torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,i):
        return self.X[i],self.y[i]


train_loader=DataLoader(
    DS(X_train,y_train),
    batch_size=256,
    shuffle=True
)

val_loader=DataLoader(
    DS(X_val,y_val),
    batch_size=512
)


class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(128*4*4,128),
            nn.ReLU(),

            nn.Linear(128,5)
        )

    def forward(self,x):
        return self.net(x)


model=CNN().to(device)

opt=torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

loss_fn=nn.CrossEntropyLoss()


for epoch in range(6):

    model.train()

    for x,t in train_loader:

        x,t=x.to(device),t.to(device)

        opt.zero_grad()

        loss=loss_fn(model(x),t)

        loss.backward()

        opt.step()


    model.eval()

    pred=[]
    true=[]

    with torch.no_grad():

        for x,t in val_loader:

            out=model(x.to(device))

            pred.extend(
                out.argmax(1).cpu().numpy()
            )

            true.extend(t.numpy())


    print(
        epoch,
        f1_score(true,pred,average="micro")
    )

(150000, 32, 32)
0 0.9028666666666667
1 0.9695333333333334
2 0.9920666666666667
3 0.9965333333333334
4 0.9980666666666667
5 0.9981333333333333


In [24]:
model.eval()

test_df = pd.read_csv(TEST_CSV)

X_test = []

for _,row in test_df.iterrows():

    img=np.array(
        Image.open(
            os.path.join(TEST_DIR,row.id)
        ).convert("RGB")
    )

    X_test.append(clean(img))


X_test=np.array(
    X_test,
    dtype=np.float32
)


test_tensor=torch.tensor(X_test).unsqueeze(1)

test_loader=DataLoader(
    test_tensor,
    batch_size=512,
    shuffle=False
)


preds=[]

with torch.no_grad():

    for x in test_loader:

        out=model(
            x.to(device)
        )

        preds.extend(
            out.argmax(1)
            .cpu()
            .numpy()
        )


submission=pd.DataFrame({
    "id":test_df.id,
    "label":preds
})


submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())
print("saved:",submission.shape)

          id  label
0  00000.png      2
1  00001.png      4
2  00002.png      4
3  00003.png      3
4  00004.png      4
saved: (5000, 2)
